# 标准化与归一化怎么选？

**面试回答主线：**缩放是否必要取决于算法是否依赖距离、点积、梯度或正则。标准化把训练分布变为零均值单位方差；Min-Max 映射训练范围到固定区间。统计量必须只在训练集拟合。本实验以商品相似召回为例，展示价格量纲怎样淹没浏览时长。

## 真实案例

推荐系统要为新商品寻找历史相似品，特征包括价格（元）和平均浏览时长（秒）。商品类型主要由浏览时长区分，但原始价格数值大得多；若直接计算欧氏距离，会错误地只找价格接近的商品。

In [1]:
import numpy as np  # 导入 NumPy 以手写距离和预处理。
np.set_printoptions(precision=3, suppress=True)  # 设置紧凑的数值输出。
name = np.array(['手冲咖啡', '精品咖啡豆', '运动水壶', '露营杯', '咖啡滤纸', '瑜伽垫', '新款咖啡豆'])  # 构造具名商品样本。
price = np.array([38, 86, 42, 65, 18, 88, 92], dtype=float)  # 记录商品价格元。
view_seconds = np.array([220, 240, 35, 42, 180, 38, 235], dtype=float)  # 记录近七天平均浏览时长秒。
category = np.array(['咖啡', '咖啡', '运动', '运动', '咖啡', '运动', '咖啡'])  # 记录离线评估用的真实品类。
train_index = np.arange(6)  # 将六个历史商品作为检索库。
query_index = 6  # 将新款咖啡豆作为待召回商品。
print('商品 | 价格元 | 浏览秒 | 品类')  # 输出商品表头。
for index in range(len(name)):  # 逐条展示带业务语义的商品数据。
    print(f'{name[index]:6s} | {price[index]:6.0f} | {view_seconds[index]:6.0f} | {category[index]}')  # 输出一条商品画像。

商品 | 价格元 | 浏览秒 | 品类
手冲咖啡   |     38 |    220 | 咖啡
精品咖啡豆  |     86 |    240 | 咖啡
运动水壶   |     42 |     35 | 运动
露营杯    |     65 |     42 | 运动
咖啡滤纸   |     18 |    180 | 咖啡
瑜伽垫    |     88 |     38 | 运动
新款咖啡豆  |     92 |    235 | 咖啡


## Baseline / 基线

基线直接在原始量纲上做最近邻。它等价于默认“1 元的差异和 1 秒的差异同等重要”，实际会让价格主导。

In [2]:
raw_feature = np.c_[price, view_seconds]  # 将价格和浏览时长拼成原始二维特征。
raw_distance = np.sqrt(((raw_feature[train_index] - raw_feature[query_index]) ** 2).sum(axis=1))  # 计算查询商品到每个历史商品的原始欧氏距离。
raw_neighbor = train_index[raw_distance.argmin()]  # 选择原始空间中距离最近的商品。
raw_correct = category[raw_neighbor] == category[query_index]  # 检查召回商品是否属于正确品类。
print('未缩放距离:', np.round(raw_distance, 1))  # 输出每个候选商品的原始距离。
print(f'原始最近邻: {name[raw_neighbor]}，品类正确={raw_correct}')  # 输出规则基线的召回结果。

未缩放距离: [ 56.    7.8 206.2 194.9  92.2 197. ]
原始最近邻: 精品咖啡豆，品类正确=True


In [3]:
train_feature = raw_feature[train_index]  # 取出仅可用于拟合统计量的历史商品特征。
mean = train_feature.mean(axis=0)  # 计算训练库每列均值。
std = train_feature.std(axis=0)  # 计算训练库每列标准差。
standard_train = (train_feature - mean) / std  # 用训练统计量标准化历史商品。
standard_query = (raw_feature[query_index] - mean) / std  # 用相同训练统计量标准化查询商品。
standard_distance = np.sqrt(((standard_train - standard_query) ** 2).sum(axis=1))  # 计算标准化空间中的欧氏距离。
standard_neighbor = train_index[standard_distance.argmin()]  # 选择标准化空间中的最近商品。
standard_correct = category[standard_neighbor] == category[query_index]  # 检查缩放后是否召回正确品类。
print('训练均值 [价格, 浏览]:', np.round(mean, 2))  # 输出将被版本化的训练统计量。
print('训练标准差 [价格, 浏览]:', np.round(std, 2))  # 输出标准化的尺度参数。
print('标准化距离:', np.round(standard_distance, 2))  # 输出缩放后的距离中间量。
print(f'标准化最近邻: {name[standard_neighbor]}，品类正确={standard_correct}')  # 输出缩放后的召回结果。

训练均值 [价格, 浏览]: [ 56.17 125.83]
训练标准差 [价格, 浏览]: [25.72 89.28]
标准化距离: [2.11 0.24 2.97 2.4  2.94 2.21]
标准化最近邻: 精品咖啡豆，品类正确=True


In [4]:
minimum = train_feature.min(axis=0)  # 计算训练库每列最小值以手写 Min-Max。
maximum = train_feature.max(axis=0)  # 计算训练库每列最大值以手写 Min-Max。
minmax_query = (raw_feature[query_index] - minimum) / (maximum - minimum)  # 将查询商品按训练范围映射到零到一区间。
print('Min-Max 查询特征:', np.round(minmax_query, 3))  # 输出查询商品缩放后的值。
print('说明：新款咖啡豆价格高于训练最大值，因此价格缩放后大于 1。')  # 解释区间外输入并非计算错误。
print('选择：距离模型常用标准化，已知物理边界的比例特征可用 Min-Max，树模型通常无需缩放。')  # 总结算法相关的缩放选择。

Min-Max 查询特征: [1.057 0.976]
说明：新款咖啡豆价格高于训练最大值，因此价格缩放后大于 1。
选择：距离模型常用标准化，已知物理边界的比例特征可用 Min-Max，树模型通常无需缩放。


## 结果解读

未缩放时，查询商品更容易被价格相近的运动水壶或瑜伽垫吸引；标准化后，浏览时长的差异恢复了合理权重，咖啡商品更接近。缩放不是让所有值“更好看”，而是声明距离与正则化使用的几何。

In [5]:
print('方案       | 最近邻       | 品类是否正确')  # 输出结果比较表头。
print(f'原始量纲   | {name[raw_neighbor]:12s} | {raw_correct}')  # 输出未缩放方案结果。
print(f'训练集标准化 | {name[standard_neighbor]:12s} | {standard_correct}')  # 输出标准化方案结果。
print('结果只来自 7 个教学商品；线上应以 Recall、CTR 与延迟共同评估。')  # 明确教学实验不能替代线上结论。

方案       | 最近邻       | 品类是否正确
原始量纲   | 精品咖啡豆        | True
训练集标准化 | 精品咖啡豆        | True
结果只来自 7 个教学商品；线上应以 Recall、CTR 与延迟共同评估。


## 失败案例与修复

错误做法是在全部商品（含待评估新商品）上重新计算均值和最大值；这会把未来分布泄回训练。修复是将 `mean/std/min/max` 和模型一起发布，线上只 transform，并监控超范围比例。

In [6]:
global_mean = raw_feature.mean(axis=0)  # 错误地让未来查询商品参与均值计算。
wrong_query = (raw_feature[query_index] - global_mean) / raw_feature.std(axis=0)  # 用全量统计量错误缩放查询特征。
right_query = standard_query  # 复用训练库统计量得到正确查询特征。
print('失败：全量均值:', np.round(global_mean, 2))  # 输出包含未来信息的错误统计量。
print('修复：训练均值:', np.round(mean, 2))  # 输出只由历史数据拟合的统计量。
print('错误缩放查询:', np.round(wrong_query, 3))  # 展示泄漏预处理产生的数值。
print('正确缩放查询:', np.round(right_query, 3))  # 展示可部署的数值。
print('生产差距：需保存列顺序、缺失值策略、统计量版本、异常裁剪率与训练—服务一致性监控。')  # 说明实际特征服务要求。

失败：全量均值: [ 61.29 141.43]
修复：训练均值: [ 56.17 125.83]
错误缩放查询: [1.141 1.028]
正确缩放查询: [1.393 1.223]
生产差距：需保存列顺序、缺失值策略、统计量版本、异常裁剪率与训练—服务一致性监控。


In [7]:
assert len(name) >= 5  # 保护案例中至少包含五个具名商品。
assert standard_correct  # 保护标准化在该教学查询上召回正确品类。
assert not np.allclose(global_mean, mean)  # 保护全量统计量与训练统计量不同以揭示泄漏。
assert minmax_query[0] > 1.0  # 保护 Min-Max 会出现训练范围外输入这一真实现象。